# **Dados e Aprendizagem Automática** 

## **kNN Classifier - Tratamento com Grid Search**

Nesta abordagem, utilizámos o pré-processamento definido anteriormente (Tratamento 2), incluindo a extração de características temporais e a limpeza das variáveis categóricas. Para as variáveis categóricas, aplicámos **OneHotEncoder** de forma a evitar relações ordinais artificiais que poderiam prejudicar o desempenho do modelo.  

Na fase de modelação, construímos um **pipeline** que combina o pré-processamento com o **KNeighborsClassifier**. Para otimizar o desempenho, aplicámos **Grid Search** sobre os principais hiperparâmetros do modelo, incluindo o número de vizinhos (`n_neighbors`), o peso das amostras (`weights`) e a métrica de distância (`metric`). Esta abordagem permite encontrar a configuração que melhor se adapta aos dados, garantindo previsões mais robustas e consistentes para a classificação do fluxo de tráfego.

**Imports necessários para este teste**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline

### **Preparation**
**Load the CSVs**

In [2]:
df_train = pd.read_csv('../../Datasets/training_data.csv', encoding='latin-1')
df_test = pd.read_csv('../../Datasets/test_data.csv', encoding='latin-1')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (6812, 14)
Test shape: (1500, 13)


**Feature Engineering (Extração da data)**

In [3]:
def extract_date_features(df):
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek # Monday=0, Sunday=6
    df['month'] = df['record_date'].dt.month
    
    # Create "Weekend" feature
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create "Rush Hour" feature (7 da manhã até às 9 da manhã e 4 da tarde ate às 7 da tarde, podemos brincar com estas horas)
    df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)
    
    return df.drop(columns=['record_date'])

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)

**Missing Values e Valores Incorretos**

In [4]:
def clean_categorical_text(df):

    # Primeiro "limpamos" a coluna 'AVERAGE CLOUDINESS'
    correcoes_erros = {
        'cï¿½u': 'ceu',      # erro especifico
        'c\u00e9u': 'ceu', # é
        'c\u00fa': 'ceu',  # ú
        'c\u00fau': 'ceu', 
        'céu': 'ceu'
    }
    # regex=True permite substituir apenas parte da frase (ex: "cï¿½u claro" -> "ceu claro")
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(correcoes_erros, regex=True)

    cloud_map = {
        'ceu claro': 'ceu_claro',
        'ceu limpo': 'ceu_claro',

        'ceu pouco nublado': 'pouco_nublado',
        'nuvens dispersas': 'pouco_nublado',
        'algumas nuvens': 'pouco_nublado',

        'nuvens quebrados': 'nublado', 
        'nuvens quebradas': 'nublado',
        'tempo nublado': 'nublado',
        'nublado': 'nublado',
    }
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(cloud_map, regex=True)
    # Tratamos também dos seus missing values
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('nan', 'unknown_cloudiness')
    
    # Depois "limpamos" também a coluna da "AVERAGE RAIN"
    rain_map = {
        'chuva fraca': 'chuva_fraca',
        'chuva leve': 'chuva_fraca',
        'aguaceiros fracos': 'chuva_fraca',
        'chuvisco fraco': 'chuva_fraca',
        'chuvisco e chuva fraca': 'chuva_fraca',
        'trovoada com chuva leve': 'chuva_fraca', 

        'chuva moderada': 'chuva_moderada',
        'chuva': 'chuva_moderada',
        'aguaceiros': 'chuva_moderada',
        'trovoada com chuva': 'chuva_moderada',

        'chuva forte': 'chuva_forte',
        'chuva de intensidade pesada': 'chuva_forte',
        'chuva de intensidade pesado': 'chuva_forte'
    }
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].replace(rain_map)
    # Tratamos também dos seus missing values
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].fillna('no_rain')
    
    #df["LUMINOSITY"] = df_train["LUMINOSITY"].replace("LOW_LIGHT", "LIGHT")
    
    return df

df_train = clean_categorical_text(df_train)
df_test = clean_categorical_text(df_test)

In [5]:
df_train["AVERAGE_SPEED_DIFF"] = df_train["AVERAGE_SPEED_DIFF"].fillna("None")

**Verificação dos valores dessas colunas agora**

In [6]:
df_test['AVERAGE_CLOUDINESS'].value_counts()

AVERAGE_CLOUDINESS
unknown_cloudiness    599
ceu_claro             372
pouco_nublado         301
nublado               228
Name: count, dtype: int64

In [7]:
df_test['AVERAGE_RAIN'].value_counts()

AVERAGE_RAIN
no_rain           1360
chuva_fraca         93
chuva_moderada      47
Name: count, dtype: int64

**Drop de Colunas Redundates** 

Considera-mo-las redundantes devido a 'CITY_NAME' conter um valor constante ("Porto") e 'AVERAGE_PRECIPITATION' consistir quase apenas em zeros.

In [8]:
cols_to_drop = ['city_name', 'AVERAGE_PRECIPITATION']
df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

**Handling categoric data**

In [9]:
# Identify categorical and numerical features

categorical_features = [
    "AVERAGE_RAIN",
    "AVERAGE_CLOUDINESS",
    "LUMINOSITY"
]

numerical_features = [
    col for col in df_train.columns
    if col not in categorical_features + ["AVERAGE_SPEED_DIFF"]
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


# ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

Categorical features: ['AVERAGE_RAIN', 'AVERAGE_CLOUDINESS', 'LUMINOSITY']
Numerical features: ['AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour']


In [10]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6812 entries, 0 to 6811
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_SPEED_DIFF       6812 non-null   object 
 1   AVERAGE_FREE_FLOW_SPEED  6812 non-null   float64
 2   AVERAGE_TIME_DIFF        6812 non-null   float64
 3   AVERAGE_FREE_FLOW_TIME   6812 non-null   float64
 4   LUMINOSITY               6812 non-null   object 
 5   AVERAGE_TEMPERATURE      6812 non-null   float64
 6   AVERAGE_ATMOSP_PRESSURE  6812 non-null   float64
 7   AVERAGE_HUMIDITY         6812 non-null   float64
 8   AVERAGE_WIND_SPEED       6812 non-null   float64
 9   AVERAGE_CLOUDINESS       6812 non-null   object 
 10  AVERAGE_RAIN             6812 non-null   object 
 11  hour                     6812 non-null   int32  
 12  day_of_week              6812 non-null   int32  
 13  month                    6812 non-null   int32  
 14  is_weekend              

In [11]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_FREE_FLOW_SPEED  1500 non-null   float64
 1   AVERAGE_TIME_DIFF        1500 non-null   float64
 2   AVERAGE_FREE_FLOW_TIME   1500 non-null   float64
 3   LUMINOSITY               1500 non-null   object 
 4   AVERAGE_TEMPERATURE      1500 non-null   float64
 5   AVERAGE_ATMOSP_PRESSURE  1500 non-null   float64
 6   AVERAGE_HUMIDITY         1500 non-null   float64
 7   AVERAGE_WIND_SPEED       1500 non-null   float64
 8   AVERAGE_CLOUDINESS       1500 non-null   object 
 9   AVERAGE_RAIN             1500 non-null   object 
 10  hour                     1500 non-null   int32  
 11  day_of_week              1500 non-null   int32  
 12  month                    1500 non-null   int32  
 13  is_weekend               1500 non-null   int64  
 14  is_rush_hour            

### **Modeling**

Select features and target

In [12]:
X_train = df_train.drop(columns=["AVERAGE_SPEED_DIFF"])
y_train = df_train["AVERAGE_SPEED_DIFF"]

In [13]:
X_test = df_test

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

Train/Test Split

In [14]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2022, stratify=y_train
)

Definir a Pipeline

In [15]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

Parâmetros da Grid Search

In [16]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

GridSearchCV

In [17]:
grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

Fit

In [18]:
grid.fit(X_tr, y_tr)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\User\Desktop\DAA-Project\daa\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [0.60947018 0.61112139 0.63075762 0.63002418 0.63956715 0.64287091
 0.6430544  0.64397149 0.64599119 0.64911146        nan 0.63094127
        nan 0.64011693        nan 0.65278035        nan 0.65700137
        nan 0.66361008]
  warnings.warn(


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'knn__metric': ['euclidean', 'manhattan'], 'knn__n_neighbors': [3, 5, ...], 'knn__weights': ['uniform', 'distance']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


Avaliar no teste

In [19]:
val_preds = grid.predict(X_val)

print("Best Parameters:", grid.best_params_)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 11, 'knn__weights': 'distance'}
Validation Accuracy: 0.6911225238444607

Classification Report:
               precision    recall  f1-score   support

        High       0.70      0.63      0.67       213
         Low       0.58      0.36      0.44       284
      Medium       0.67      0.73      0.70       330
        None       0.71      0.90      0.80       440
   Very_High       0.86      0.70      0.77        96

    accuracy                           0.69      1363
   macro avg       0.70      0.66      0.67      1363
weighted avg       0.68      0.69      0.68      1363



Treinar o modelo

In [20]:
best_knn = grid.best_estimator_
best_knn.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Predicts

In [21]:
preds = best_knn.predict(X_test)

Para submeter no kaggle

In [22]:
submission = pd.DataFrame({
    "RowId": range(1, len(preds)+1),
    "Speed_Diff": preds
})

submission.to_csv("../../results/kNN/knn2grid.csv", index=False)
submission.head()

,RowId,Speed_Diff
0,1,None
1,2,None
2,3,None
3,4,Medium
4,5,None
